# Tools

Implementación de un tool básico.

El modelo no sabe qué día es hoy: su conocimiento tiene fecha de corte y no lee
ningún reloj. Le vamos a dar una función que sí lo hace y vamos a correr el
ciclo completo: describir, pedir, ejecutar, devolver.

**Necesitas una API key.** Cópiala en el archivo `.env` como `OPENROUTER_API_KEY`.

In [ ]:
import os
import json
from datetime import date

import httpx
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "Falta OPENROUTER_API_KEY"

## El tool y su definición

`fecha_actual()` es una función normal de Python. El modelo nunca la ejecuta:
tú la ejecutas.

Para que el modelo sepa que existe, la describimos en el arreglo `tools` con
tres campos:

1. `name`: cómo se llama la función
2. `description`: cuándo usarla. El modelo lee esto para decidir
3. `parameters`: qué datos necesita, en JSON Schema

Esta función no necesita datos, así que `properties` va vacío.

In [ ]:
def fecha_actual():
    return date.today().strftime("%d/%m/%Y")


tools = [
    {
        "type": "function",
        "function": {
            "name": "fecha_actual",
            "description": "Devuelve la fecha de hoy",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    }
]

print(fecha_actual())

## Primera llamada: el modelo pide la herramienta

Es el mismo `POST /chat/completions` de la clase pasada, más el campo `tools`
en el body.

El modelo no contesta con texto. Contesta con `finish_reason: "tool_calls"` y
con un `tool_call` que dice qué función quiere usar y con qué argumentos.

In [ ]:
mensajes = [{"role": "user", "content": "¿Qué fecha es hoy?"}]

r = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": mensajes,
        "tools": tools,
    },
    timeout=60,
)
respuesta = r.json()

print(respuesta["choices"][0]["finish_reason"])
print(json.dumps(respuesta["choices"][0]["message"]["tool_calls"], indent=2, ensure_ascii=False))

## Ejecutar y devolver

Los argumentos llegan como texto JSON, así que los pasamos por `json.loads`.
Ejecutamos la función nosotros y agregamos dos turnos a `mensajes`:

1. El mensaje del assistant, tal como vino, con su `tool_calls`
2. Un turno nuevo con `role: "tool"`, el `tool_call_id` de esa llamada y el
   resultado en `content`

Con la lista así de completa hacemos la segunda llamada y ahora sí el modelo
responde con texto.

In [ ]:
llamada = respuesta["choices"][0]["message"]["tool_calls"][0]
args = json.loads(llamada["function"]["arguments"])
print(args)

resultado = fecha_actual()
print(resultado)

mensajes.append(respuesta["choices"][0]["message"])
mensajes.append(
    {
        "role": "tool",
        "tool_call_id": llamada["id"],
        "content": resultado,
    }
)

r = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": mensajes,
        "tools": tools,
    },
    timeout=60,
)

print(r.json()["choices"][0]["message"]["content"])